In [1]:
import json
import numpy as np
from scipy.stats import binomtest, false_discovery_control
import pandas as pd

In [2]:
#func create_paths_str(...) -> file_path

def create_paths_str(model_name, variant_name, prompt_type, dataset_type):
    return f"outputs/{model_name}/{variant_name}/responses_{prompt_type}_synthetic_dataset_{variant_name}_{dataset_type}.json"

In [3]:
#func assemble_file_paths(...) -> file_paths

def assemble_file_paths(model_name, variant_names, prompt_type):
    file_paths = []
    for dataset_type in ["gold", "random"]:
        path_list = []
        for variant in variant_names:
            path_list.append(create_paths_str(model_name, variant, prompt_type, dataset_type))
        file_paths.append(path_list)
    return file_paths

In [23]:
#func extract_output_results (file_path) -> grades

def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    #print(file_path)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades

In [5]:
#func collect_grades (file_paths) -> grades
#   file_paths.type = list; file_paths.shape = (2, a); a = number of dataset variants
#   grades.type = np.array; grades.shape = (2, b); b = a * n; n = length of each grades list

def collect_grades(file_paths):
    grades = []
    for i in range(2):
        grades_list = []
        for j in range(len(file_paths[0])):
            grades_list.extend(extract_output_results(file_paths[i][j]))
        grades.append(grades_list)
    return np.array(grades)

In [6]:
#func calc_test_statistics (grades) -> n12, n21, n_star, z0
#   grades.type = np.array; grades.shape = (2, b)
#   n12smaller = bool (if True, the alternative hypothesis H_a is "n12 < n21". If False H_a = "n12 > n21".)

def calc_test_statistics (grades, n12smaller = True):

    grades_gold, grades_random = grades
    n12 = 0; n21 = 0
    for i in range(len(grades_gold)):
        n12 += grades_gold[i] and not grades_random[i]
        n21 += not grades_gold[i] and grades_random[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)  if n_star > 0 else  0
    #n_test = n21  if n12smaller else  n12   # I.e. if H_a = "n12 < n21"
    p_value = binomtest(n21, n_star, alternative='greater').pvalue  if n_star > 0 else  1
    return n12, n21, n_star, z0, p_value

In [7]:
def do_benjamini_hochberg(p_values, sig_level):
    sort_order = np.argsort(p_values)
    p_sorted = np.sort(p_values)
    count = len(p_values)
    control_line = np.linspace(sig_level, 0, count, endpoint=False)[::-1]
    rejection_boundary = np.max(np.argwhere(p_sorted <= control_line).T[0])

    rejections = np.full(count, False)
    rejections[sort_order[:rejection_boundary+1]] = True

    return rejections

In [8]:
#func run_analysis (models, prompting_types, datasets) -> results_table
#   models.type = list of strings: same with prompting_types and datasets
#   results_table = [[models], [prompting_types], [n12], [n21], [n_star], [z0], [p_value], [rejections]]

def run_analysis(models, prompting_types, datasets, bh='own', n12smaller=True):
    results_table = {
        "model":[],
        "prompting_method":[],
        "n12":[],
        "n21":[],
        "n*":[],
        "z-stat":[],
        "p-value":[],
        "reject":[],
    }
    for model in models:
        for prompt_type in prompting_types:
            file_paths = assemble_file_paths(model, datasets, prompt_type)
            grades = collect_grades(file_paths)
            n12, n21, n_star, z0, p_value = calc_test_statistics(grades, n12smaller)
            #print(p_value)

            results_table["model"].append(model)
            results_table["prompting_method"].append(prompt_type)
            results_table["n12"].append(n12)
            results_table["n21"].append(n21)
            results_table["n*"].append(n_star)
            results_table["z-stat"].append(z0)
            results_table["p-value"].append(p_value)      
    if bh == 'own' :
        results_table["reject"] = do_benjamini_hochberg(results_table["p-value"], 0.05)
    elif bh == 'scipy' :
        results_table["p-value"] = false_discovery_control(results_table["p-value"])
        results_table["reject"] = results_table["p-value"] < 0.05
    else:
        raise KeyError(f"bh == {bh} not implemented!")

    return results_table   

#### Test Cases

##### Hypothesis 1

In [9]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    #"linda_original",
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
results_table = run_analysis(models, prompting_types, datasets)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 50/54


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,baseline,4,123,127,10.559542,0.000000,True
1,gpt-3.5-turbo,zs_cot,18,169,187,11.042214,0.000000,True
2,gpt-3.5-turbo,os,3,94,97,9.239650,0.000000,True
3,gpt-3.5-turbo,os_cot,16,114,130,8.595169,0.000000,True
4,gpt-3.5-turbo,fs,11,103,114,8.616589,0.000000,True
5,gpt-3.5-turbo,fs_cot,6,140,146,11.089919,0.000000,True
6,gpt-4-turbo,baseline,7,273,280,15.896541,0.000000,True
7,gpt-4-turbo,zs_cot,6,250,256,15.250000,0.000000,True
8,gpt-4-turbo,os,0,53,53,7.280110,0.000000,True
9,gpt-4-turbo,os_cot,1,72,73,8.309921,0.000000,True


##### Hypothesis 2

In [10]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "os_bob",
    "os_bob_cot"
]
datasets = [
    #"linda_original",
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
    "linda_variant_four",
]
results_table = run_analysis(models, prompting_types, datasets, n12smaller=True)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 18/18


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,os_bob,16,194,210,12.283167,0.000000,True
1,gpt-3.5-turbo,os_bob_cot,21,170,191,10.781262,0.000000,True
2,gpt-4-turbo,os_bob,2,128,130,11.050931,0.000000,True
3,gpt-4-turbo,os_bob_cot,4,145,149,11.551170,0.000000,True
4,gpt-4o,os_bob,0,20,20,4.472136,0.000001,True
5,gpt-4o,os_bob_cot,6,101,107,9.183997,0.000000,True
6,llama-2-70b-chat,os_bob,18,178,196,11.428571,0.000000,True
7,llama-2-70b-chat,os_bob_cot,24,204,228,11.920791,0.000000,True
8,meta-llama-3-70b-instruct,os_bob,14,161,175,11.112156,0.000000,True
9,meta-llama-3-70b-instruct,os_bob_cot,22,170,192,10.680980,0.000000,True


##### Hypothesis 3

In [11]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "linda_variant_four",
]
results_table = run_analysis(models, prompting_types, datasets)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 19/54


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,baseline,9,19,28,1.889822,0.043579,False
1,gpt-3.5-turbo,zs_cot,14,23,37,1.479591,0.093871,False
2,gpt-3.5-turbo,os,7,11,18,0.942809,0.240341,False
3,gpt-3.5-turbo,os_cot,12,13,25,0.200000,0.500000,False
4,gpt-3.5-turbo,fs,3,17,20,3.130495,0.001288,True
5,gpt-3.5-turbo,fs_cot,12,20,32,1.414214,0.107664,False
6,gpt-4-turbo,baseline,5,21,26,3.137858,0.001247,True
7,gpt-4-turbo,zs_cot,1,19,20,4.024922,0.000020,True
8,gpt-4-turbo,os,0,6,6,2.449490,0.015625,True
9,gpt-4-turbo,os_cot,1,5,6,1.632993,0.109375,False


##### Hypothesis 4

In [12]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    #"meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "sets_original",
]
results_table = run_analysis(models, prompting_types, datasets)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 4/48


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,baseline,49,1,50,-6.788225,1.000000,False
1,gpt-3.5-turbo,zs_cot,46,7,53,-5.357062,1.000000,False
2,gpt-3.5-turbo,os,7,3,10,-1.264911,0.945312,False
3,gpt-3.5-turbo,os_cot,28,28,56,0.000000,0.553073,False
4,gpt-3.5-turbo,fs,4,1,5,-1.341641,0.968750,False
5,gpt-3.5-turbo,fs_cot,13,25,38,1.946657,0.036476,False
6,gpt-4-turbo,baseline,56,0,56,-7.483315,1.000000,False
7,gpt-4-turbo,zs_cot,97,3,100,-9.400000,1.000000,False
8,gpt-4-turbo,os,49,0,49,-7.000000,1.000000,False
9,gpt-4-turbo,os_cot,47,2,49,-6.428571,1.000000,False


##### Hypothesis 5

In [13]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "sets_original_framing",
]
results_table = run_analysis(models, prompting_types, datasets)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 32/54


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,baseline,20,60,80,4.472136,0.000004,True
1,gpt-3.5-turbo,zs_cot,26,61,87,3.752394,0.000112,True
2,gpt-3.5-turbo,os,16,23,39,1.120897,0.168392,False
3,gpt-3.5-turbo,os_cot,42,44,86,0.215666,0.457106,False
4,gpt-3.5-turbo,fs,22,31,53,1.236245,0.135840,False
5,gpt-3.5-turbo,fs_cot,27,27,54,0.000000,0.554038,False
6,gpt-4-turbo,baseline,30,24,54,-0.816497,0.829555,False
7,gpt-4-turbo,zs_cot,27,19,46,-1.179536,0.908038,False
8,gpt-4-turbo,os,29,38,67,1.099525,0.164216,False
9,gpt-4-turbo,os_cot,30,46,76,1.835326,0.042323,False


##### Hypothesis 6

In [24]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "llama-2-70b-chat",
    "meta-llama-3-70b-instruct",
    #"meta-llama-3-8b-instruct",
    #"claude-3-opus-20240229",
    #"claude-3-sonnet-20240229",
    #"mistral-large-latest",
]
prompting_types = [
    "weak_control_zs_cot",
    "control_zs_cot",
    "weak_control_os_cot",
    "control_os_cot",
]
datasets = [
    "linda_original",
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
    "linda_variant_four",
    "sets_original",
]
results_table = run_analysis(models, prompting_types, datasets, n12smaller=True)
print(f"rejected: {results_table['reject'].sum()}/{len(results_table['reject'])}")
df = pd.DataFrame(results_table)
df.round(6)

rejected: 4/20


,model,prompting_method,n12,n21,n*,z-stat,p-value,reject
0,gpt-3.5-turbo,weak_control_zs_cot,84,107,191,1.664222,0.055588,False
1,gpt-3.5-turbo,control_zs_cot,63,57,120,-0.547723,0.738505,False
2,gpt-3.5-turbo,weak_control_os_cot,76,77,153,0.080845,0.500000,False
3,gpt-3.5-turbo,control_os_cot,28,21,49,-1.000000,0.873565,False
4,gpt-4-turbo,weak_control_zs_cot,26,35,61,1.152332,0.152839,False
5,gpt-4-turbo,control_zs_cot,39,3,42,-5.554921,1.000000,False
6,gpt-4-turbo,weak_control_os_cot,26,8,34,-3.086975,0.999589,False
7,gpt-4-turbo,control_os_cot,15,1,16,-3.500000,0.999985,False
8,gpt-4o,weak_control_zs_cot,9,11,20,0.447214,0.411901,False
9,gpt-4o,control_zs_cot,8,8,16,0.000000,0.598190,False


#### Tests

In [19]:
np.linspace(1, 0, 20, endpoint=False)[::-1] * 0.05

array([0.0025, 0.005 , 0.0075, 0.01  , 0.0125, 0.015 , 0.0175, 0.02  ,
       0.0225, 0.025 , 0.0275, 0.03  , 0.0325, 0.035 , 0.0375, 0.04  ,
       0.0425, 0.045 , 0.0475, 0.05  ])

In [20]:
random_list = np.sort(np.random.rand(20) * 0.06)
linear_list = np.linspace(0.05, 0, 20, endpoint=False)[::-1]
print(random_list)
print(linear_list)
where_bigger = np.argwhere(random_list <= linear_list).T[0]
print(where_bigger)
np.max(where_bigger)

[0.00045849 0.0043322  0.00558155 0.01441019 0.01448306 0.0180893
 0.02196842 0.02214146 0.0250113  0.02804885 0.03038087 0.0330408
 0.03390001 0.03890574 0.04451873 0.04635931 0.05174444 0.05453214
 0.05602844 0.05786526]
[0.0025 0.005  0.0075 0.01   0.0125 0.015  0.0175 0.02   0.0225 0.025
 0.0275 0.03   0.0325 0.035  0.0375 0.04   0.0425 0.045  0.0475 0.05  ]
[0 1 2]


2

#### v1
make basic analysis work

In [ ]:
import json
import numpy as np
from scipy.stats import binomtest

In [ ]:
file_1 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json"
file_2 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json"

In [ ]:
#func extract_output_results (file_1, file_2) -> grades_1, grades_2

def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades

grades_1 = extract_output_results(file_1)
grades_2 = extract_output_results(file_2)

In [ ]:
#func calc_test_statistics (grades_1, grades_2) -> n12, n21, n_star, z0

def calc_test_statistics (grades_1, grades_2):
    n12 = 0; n21 = 0
    for i in range(len(grades_1)) :
        n12 += grades_1[i] and not grades_2[i]
        n21 += not grades_1[i] and grades_2[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue

    return n12, n21, n_star, z0, p_value

calc_test_statistics(grades_1, grades_2)

(1, 45, 46, 6.487446070815474, 6.679101716144942e-13)

#### v2
make analysis work with the whole set of dataset variants

In [ ]:
import json
import numpy as np
from scipy.stats import binomtest

In [ ]:
#func assemble_file_paths(...) -> file_paths
def assemble_file_paths(model_name, variant_names, prompt_type):
    file_paths = []
    for dataset_type in ["gold", "random"]:
        path_list = []
        for variant in variant_names:
            path_list.append(create_paths_str(model_name, variant, prompt_type, dataset_type))
        file_paths.append(path_list)
    return file_paths



#func create_paths_str(...) -> file_path
def create_paths_str(model_name, variant_name, prompt_type, dataset_type):
    return f"outputs/{model_name}/{variant_name}/responses_{prompt_type}_synthetic_dataset_{variant_name}_{dataset_type}.json"



model_name = "claude-3-opus-20240229"
variant_names = [
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
prompt_type = "baseline"
file_paths = assemble_file_paths(model_name, variant_names, prompt_type)
file_paths

[['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_dataset_linda_variant_three_gold.json'],
 ['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_d

In [ ]:
#func collect_grades (file_paths) -> grades
#   file_paths.type = list; file_paths.shape = (2, a); a = number of dataset variants
#   grades.type = np.array; grades.shape = (2, b); b = a * n; n = length of each grades list
def collect_grades(file_paths):
    grades = []
    for i in range(2):
        grades_list = []
        for j in range(len(file_paths[0])):
            grades_list.extend(extract_output_results(file_paths[i][j]))
        grades.append(grades_list)
    return np.array(grades)



#func extract_output_results (file_path) -> grades
def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    #print(file_path)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades



grades = collect_grades(file_paths)
grades.shape

(2, 400)

In [ ]:
#func calc_test_statistics (grades) -> n12, n21, n_star, z0
#   grades.type = np.array; grades.shape = (2, b)
#   n12smaller = bool (if True, the alternative hypothesis H_a is "n12 < n21". If False H_a = "n12 > n21".)

def calc_test_statistics (grades, n12smaller = True):

    grades_gold, grades_random = grades
    n12 = 0; n21 = 0
    for i in range(len(grades_gold)):
        n12 += grades_gold[i] and not grades_random[i]
        n21 += not grades_gold[i] and grades_random[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)  if n_star > 0 else  0
    #n_test = n21  if n12smaller else  n12   # I.e. if H_a = "n12 < n21"
    p_value = binomtest(n21, n_star, alternative='greater').pvalue  if n_star > 0 else  1
    return n12, n21, n_star, z0, p_value



calc_test_statistics(grades)

(16, 175, 191, 11.504836224149503, 2.7525638867682486e-35)

#### v3
make the analysis work with all models and prompting methods

In [ ]:
import json
import numpy as np
from scipy.stats import binomtest

In [ ]:
#func run_analysis (models, prompting_types, datasets) -> results_table
#   models.type = list of strings: same with prompting_types and datasets
#   results_table = [[models], [prompting_types], [n12], [n21], [n_star], [z0], [p_value], [rejections]]

def run_analysis(models, prompting_types, datasets):
    results_table = [[], [], [], [], [], [], [], [], []]
    for model in models:
        for prompt_type in prompting_types:
            file_paths = assemble_file_paths(model, datasets, prompt_type)
            grades = collect_grades(file_paths)
            n12, n21, n_star, z0, p_value = calc_test_statistics(grades)

            results_table[0].append(model)
            results_table[1].append(prompt_type)
            results_table[2].append(n12)
            results_table[3].append(n21)
            results_table[4].append(n_star)
            results_table[5].append(z0)
            results_table[6].append(p_value)
            

    # rejections
    #results_table[7].extend(np.full(len(results_table[0]), False))
    results_table[7] = do_benjamini_hochberg(results_table[6], 0.05)

    return results_table



def do_benjamini_hochberg(p_values, sig_level):
    sort_order = np.argsort(p_values)
    p_sorted = np.sort(p_values)
    count = len(p_values)
    control_line = np.linspace(sig_level, 0, count, endpoint=False)[::-1]
    rejection_boundary = np.max(np.argwhere(p_sorted <= control_line).T[0])

    rejections = np.full(count, False)
    rejections[sort_order[:rejection_boundary+1]] = True

    return rejections
    


models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "llama-2-70b-chat",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
results_table = run_analysis(models, prompting_types, datasets)
print(results_table[7])

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True False  True  True  True False  True False  True
  True  True  True  True  True  True  True  True  True  True False  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]


In [ ]:
models = [
    "gpt-3.5-turbo",
    "gpt-4-turbo",
    "gpt-4o",
    "meta-llama-3-70b-instruct",
    "meta-llama-3-8b-instruct",
    "llama-2-70b-chat",
    "claude-3-opus-20240229",
    "claude-3-sonnet-20240229",
    "mistral-large-latest",
]
prompting_types = [
    "baseline",
    "zs_cot",
    "os",
    "os_cot",
    "fs",
    "fs_cot",
]
datasets = [
    "linda_variant_four",
]
results_table = run_analysis(models, prompting_types, datasets)
print(np.array(results_table[2:5]).T)

[[ 9 19 28]
 [14 23 37]
 [ 7 11 18]
 [12 13 25]
 [ 3 17 20]
 [12 20 32]
 [ 5 21 26]
 [ 1 19 20]
 [ 0  6  6]
 [ 1  5  6]
 [ 2  8 10]
 [ 3  7 10]
 [ 3 19 22]
 [ 1  7  8]
 [ 0  1  1]
 [ 2  2  4]
 [ 0  0  0]
 [ 2  4  6]
 [ 4 20 24]
 [ 3 27 30]
 [ 3  6  9]
 [ 5  4  9]
 [ 6  6 12]
 [ 8  9 17]
 [ 4 18 22]
 [ 9 25 34]
 [ 1  2  3]
 [ 9 19 28]
 [ 4  6 10]
 [13 13 26]
 [ 7  6 13]
 [10 14 24]
 [ 3 17 20]
 [ 9 20 29]
 [ 8 16 24]
 [12 11 23]
 [ 6 24 30]
 [ 7 28 35]
 [ 1  7  8]
 [ 4  7 11]
 [ 5 12 17]
 [ 7 17 24]
 [ 4 33 37]
 [11 25 36]
 [ 1  9 10]
 [ 5 10 15]
 [ 6 17 23]
 [ 7 13 20]
 [ 3 22 25]
 [ 2 25 27]
 [ 0  3  3]
 [ 0  4  4]
 [ 2 12 14]
 [ 5 11 16]]


In [ ]:
import pandas as pd



#### v4

In [ ]:
import json
import numpy as np
from scipy.stats import binomtest, false_discovery_control
import pandas as pd

In [ ]:
#func run_analysis (models, prompting_types, datasets) -> results_table
#   models.type = list of strings: same with prompting_types and datasets
#   results_table = [[models], [prompting_types], [n12], [n21], [n_star], [z0], [p_value], [rejections]]

def run_analysis(models, prompting_types, datasets, bh='own', n12smaller=True):
    results_table = {
        "model":[],
        "prompting_method":[],
        "n12":[],
        "n21":[],
        "n*":[],
        "z-stat":[],
        "p-value":[],
        "reject":[],
    }
    for model in models:
        for prompt_type in prompting_types:
            file_paths = assemble_file_paths(model, datasets, prompt_type)
            grades = collect_grades(file_paths)
            n12, n21, n_star, z0, p_value = calc_test_statistics(grades, n12smaller)
            #print(p_value)

            results_table["model"].append(model)
            results_table["prompting_method"].append(prompt_type)
            results_table["n12"].append(n12)
            results_table["n21"].append(n21)
            results_table["n*"].append(n_star)
            results_table["z-stat"].append(z0)
            results_table["p-value"].append(p_value)      
    if bh == 'own' :
        results_table["reject"] = do_benjamini_hochberg(results_table["p-value"], 0.05)
    elif bh == 'scipy' :
        results_table["p-value"] = false_discovery_control(results_table["p-value"])
        results_table["reject"] = results_table["p-value"] < 0.05
    else:
        raise KeyError(f"bh == {bh} not implemented!")

    return results_table



def do_benjamini_hochberg(p_values, sig_level):
    sort_order = np.argsort(p_values)
    p_sorted = np.sort(p_values)
    count = len(p_values)
    control_line = np.linspace(sig_level, 0, count, endpoint=False)[::-1]
    rejection_boundary = np.max(np.argwhere(p_sorted <= control_line).T[0])

    rejections = np.full(count, False)
    rejections[sort_order[:rejection_boundary+1]] = True

    return rejections
   